# Unit 5 solutions: steer like a driver looks ahead

These reference solutions build pure pursuit and Stanley from their geometry, predict each answer first where the lesson gives a law (the circle, the delay's blind ticks, the linearized gains, Stanley on a straight), and then measure it.
Every race is seeded, so every number below is reproducible, and executable asserts check each answer against an independent derivation.
Every number is a number about racekit's **kinematic** car: a model, not the truth.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import KinematicCar
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import run_race
from racekit.tracks import load_track

AGENTS = Path("../../projects/racekit/agents")
UNIT_04, UNIT_05 = AGENTS / "unit-04.py", AGENTS / "unit-05.py"
workdir = Path(mkdtemp(prefix="roar-unit-05-sol-"))

# racekit's kinematic car (a model, not the truth)
WHEELBASE = 2.875  # m, L
DELTA_MAX = math.radians(70.0)  # the wheel angle at |steer| = 1; delta = -steer * DELTA_MAX
TICK = 0.05 + 1e-9  # race times are pinned to one control tick
RMS_TOL = 0.0005  # m, the book's tolerance on an RMS cross-track error


def rms_offset(result):
    # RMS cross-track error over the driving rows (held == 0), as in Unit 4.
    telemetry = result.telemetry
    offset = telemetry["offset"][telemetry["held"] == 0]
    return float(np.sqrt(np.mean(offset ** 2)))


def clean(result):
    # finished, with no collisions
    return result.finished and result.collisions == 0

## Exercise 1 - Pure pursuit by hand

The frame change rotates the offset (dx, dy) by −yaw: forward = cos ψ·dx + sin ψ·dy, left = −sin ψ·dx + cos ψ·dy.
The arc tangent to the heading through a point at bearing α and distance d has d = 2R sin α (the tangent–chord angle), so κ = 2 sin α / d, and δ = arctan(L κ) = atan2(2L sin α, d).

**Prediction.** The car at (12, −3) heads 150°, up and to the left; the target (2, 4) is offset (−10, 7), whose direction is about 145°, so it lies about 5° to the **right** of the heading, about 12.2 m away.
A target on the right gives α < 0, κ < 0, δ < 0 and a **positive** steer (a positive steer turns right).
On a circle the target lies on the arc, so κ·R must be exactly 1 for every R and chord.

In [ ]:
def to_car_frame(px, py, cx, cy, yaw):
    dx, dy = px - cx, py - cy
    c, s = math.cos(yaw), math.sin(yaw)
    return c * dx + s * dy, -s * dx + c * dy


def pursuit_curvature(alpha, distance):
    return 2.0 * math.sin(alpha) / distance


def pursuit_wheel_angle(alpha, distance):
    return math.atan2(2.0 * WHEELBASE * math.sin(alpha), distance)


forward, left = to_car_frame(2.0, 4.0, 12.0, -3.0, math.radians(150.0))
alpha = math.atan2(left, forward)
distance = math.hypot(forward, left)
kappa = pursuit_curvature(alpha, distance)
delta = pursuit_wheel_angle(alpha, distance)
steer = float(np.clip(-delta / DELTA_MAX, -1.0, 1.0))
turns_left = kappa > 0.0


def on_circle(radius, chord):
    # The point at straight-line distance `chord` along the left circle of radius R centred at (0, R),
    # starting from the origin heading along +x: the central angle theta has chord = 2 R sin(theta / 2).
    half = chord / 2.0
    theta = 2.0 * math.atan2(half, math.sqrt(radius * radius - half * half))
    return radius * math.sin(theta), radius - radius * math.cos(theta)


circle_products = []
for radius in (15.0, 30.0, 60.0):
    tx, ty = on_circle(radius, 10.0)
    f, l = to_car_frame(tx, ty, 0.0, 0.0, 0.0)
    circle_products = circle_products + [pursuit_curvature(math.atan2(l, f), math.hypot(f, l)) * radius]

print(f"target in the car frame: forward {forward:.4f} m, left {left:+.4f} m")
print(f"alpha = {alpha:+.5f} rad ({math.degrees(alpha):+.2f} deg), d = {distance:.4f} m")
print(f"kappa = {kappa:+.6f} 1/m, delta = {delta:+.6f} rad, steer = {steer:+.5f}, turns left: {turns_left}")
print("kappa * R on the circles:", circle_products)

# independent checks: the target's world bearing minus the heading, and the arc through the target
world_bearing = math.atan2(4.0 - (-3.0), 2.0 - 12.0)
assert abs(alpha - (world_bearing - math.radians(150.0))) < 1e-12
assert abs(distance - math.hypot(10.0, 7.0)) < 1e-12
assert abs(math.tan(delta) / WHEELBASE - kappa) < 1e-12
assert alpha < 0.0 and steer > 0.0 and not turns_left
assert abs(steer - 0.03353) < 1e-5
assert len(circle_products) == 3 and max(abs(p - 1.0) for p in circle_products) < 1e-12

## Exercise 2 - Where the car looks

**Prediction.** L_d = 0.3·v gives 1.5, 3, 6, 9, 12 and 45 m, clipped to [3, 40]: 3, 3, 6, 9, 12 and **40** m.
`test-circuit` starts on a straight with waypoints about 2 m apart and the car on waypoint 0, so the target is the first waypoint at least L_d ahead: index ⌈L_d / 2⌉, i.e. 2, 2, 3 (or 4 if the spacing is just under 2 m), 5, 6 or 7, and 20 or 21; the exact spacing decides the boundary cases, so measure it.

The clip **changes** L_d only at 5 m/s (1.5 → 3) and at 150 m/s (45 → 40).
At 10 m/s, k·v = 3.0 m is exactly L_min, so the clip changes nothing there and, by the statement's rule, 10 m/s is not limited.

In [ ]:
def lookahead_distance(speed, k=0.3, minimum=3.0, maximum=40.0):
    return min(max(k * speed, minimum), maximum)


def first_target(xs, ys, segment, px, py, lookahead):
    n = len(xs)
    index = segment
    for _ in range(n):
        index = (index + 1) % n
        if math.hypot(xs[index] - px, ys[index] - py) >= lookahead:
            return index
    return index


track = load_track("test-circuit")
xs, ys = track.x, track.y
px, py, yaw = float(track.x[0]), float(track.y[0]), float(track.yaw[0])  # on waypoint 0, facing along the track
SPEEDS = (5.0, 10.0, 20.0, 30.0, 40.0, 150.0)
lookaheads = {speed: lookahead_distance(speed) for speed in SPEEDS}
target_index = {speed: first_target(xs, ys, 0, px, py, lookaheads[speed]) for speed in SPEEDS}
limited_speeds = sorted([speed for speed in SPEEDS if lookaheads[speed] != 0.3 * speed])
gaps = np.hypot(np.diff(np.r_[xs, xs[:1]]), np.diff(np.r_[ys, ys[:1]]))
spacing = float(np.median(gaps))
why_distance = (
    "The lookahead is a distance the car should look ahead, so the target must be chosen by distance: "
    "a fixed number of waypoints ahead would mean a different distance on every track generator or "
    "in CARLA (whose waypoints need not be 2 m apart, or evenly spaced), silently changing the "
    "controller's gains."
)

for speed in SPEEDS:
    k = target_index[speed]
    print(f"v = {speed:5.0f} m/s: L_d = {lookaheads[speed]:4.1f} m, target {k:3d} "
          f"at {math.hypot(xs[k] - px, ys[k] - py):.3f} m")
print("limited at:", limited_speeds, f"| median spacing {spacing:.6f} m (min {gaps.min():.4f}, max {gaps.max():.4f})")

assert lookaheads == {5.0: 3.0, 10.0: 3.0, 20.0: 6.0, 30.0: 9.0, 40.0: 12.0, 150.0: 40.0}
assert limited_speeds == [5.0, 150.0]
for speed in SPEEDS:
    k, ld = target_index[speed], lookaheads[speed]
    # the target is far enough, and the waypoint before it (still after segment 0) is not
    assert math.hypot(xs[k] - px, ys[k] - py) >= ld
    assert k == 1 or math.hypot(xs[k - 1] - px, ys[k - 1] - py) < ld
assert target_index == {5.0: 2, 10.0: 2, 20.0: 4, 30.0: 5, 40.0: 7, 150.0: 21}
assert abs(spacing - 2.0) < 0.002

## Exercise 3 - A delay, measured

The balancing throttle is 4·10⁻⁴·30² / 6 = 0.06.
**Prediction.** The FIFO starts with N zero actions, so for the first N ticks the car drives straight (and coasts: the throttle is late too).
The yaw first changes **after tick N + 1**, and the car has driven about N·30·0.05 = 1.5·N m blind (a little less: it coasts).
Once the steer arrives the car runs on the circle R = L / tan(0.05·70°) ≈ 47.0 m at 30 m/s; over the full second (no delay) it turns 30 / 47 ≈ 0.64 rad and moves y ≈ −R(1 − cos 0.64) ≈ −9.25 m.
With N ticks of delay it turns for only 20 − N ticks, so the lateral move shrinks by roughly 0.8–0.9 m per tick of delay.

In [ ]:
C_DRAG, A_THROTTLE = 4.0e-4, 6.0
throttle = C_DRAG * 30.0 ** 2 / A_THROTTLE

first_response = {}
blind_distance = {}
lateral_after_1s = {}
for lag in range(5):
    car = KinematicCar(lag_ticks=lag)
    car.reset(0.0, 0.0, 0.0, 30.0)
    poses = np.zeros((20, 3))
    for tick in range(20):
        car.advance(throttle, 0.05, 0.0, 0.0, 0.0, 0.05, 10)
        poses[tick] = (car.x, car.y, car.yaw)
    first = int(np.nonzero(poses[:, 2])[0][0]) + 1
    first_response[lag] = first
    blind_distance[lag] = 0.0 if first == 1 else float(math.hypot(poses[first - 2, 0], poses[first - 2, 1]))
    lateral_after_1s[lag] = float(poses[-1, 1])
lag_cost = {lag: lateral_after_1s[lag] - lateral_after_1s[0] for lag in range(5)}
why_it_matters = (
    "Each correction acts on an error that is N ticks old, so for N ticks the controller keeps pushing "
    "at full strength in the direction that was right N ticks ago; it overshoots, the next correction "
    "is larger and also late, and a tight controller can oscillate with a growing amplitude until it crashes."
)

for lag in range(5):
    print(f"N = {lag}: first response after tick {first_response[lag]}, blind {blind_distance[lag]:.4f} m, "
          f"y after 1 s {lateral_after_1s[lag]:+.4f} m, cost {lag_cost[lag]:+.4f} m")

radius = WHEELBASE / math.tan(0.05 * DELTA_MAX)
assert abs(throttle - 0.06) < 1e-12
assert first_response == {lag: lag + 1 for lag in range(5)}
for lag in range(5):
    # coasting for N ticks: a little less than N * 1.5 m
    assert lag * 1.5 - 0.02 < blind_distance[lag] <= lag * 1.5
# no delay: the steady circle at 30 m/s for 1 s
assert abs(lateral_after_1s[0] - (-radius * (1.0 - math.cos(30.0 / radius)))) < 1e-3
assert all(lag_cost[lag] > lag_cost[lag - 1] for lag in range(1, 5))

## Exercise 4 - Pure pursuit is a PD in disguise

With ψ = 0 the car frame is the world frame shifted by e: the target (L_d, 0) is at forward L_d, left −e, so sin α = −e / √(L_d² + e²) and d = √(L_d² + e²).
**Prediction.** The exact δ = atan2(−2L e / d, d) = arctan(−2L e / (L_d² + e²)), while the linear law is −2L e / L_d².
For small e the relative error is about e² / L_d² (from the denominator) plus a little from the arctan: about 0.0005% at 0.02 m, 0.01% at 0.1 m, 0.35% at 0.5 m and over 5% at 2 m, so the largest e below 1% is **0.5 m**.

The gains: K_E = 2L / L_d², K_HEAD = 2L / L_d, with L_d = clip(0.3 v, 3, 40) = 3, 6, 9, 12 m at 10, 20, 30, 40 m/s.
K_E = 0.2 gives L_d = √(2L / 0.2) = √(10 L) ≈ 5.362 m, inside [3, 40], so v = √(10 L) / 0.3 ≈ 17.87 m/s.

In [ ]:
def equivalent_k_e(lookahead):
    return 2.0 * WHEELBASE / lookahead ** 2


def equivalent_k_head(lookahead):
    return 2.0 * WHEELBASE / lookahead


E_VALUES = np.array([0.02, 0.1, 0.5, 2.0])
LD = 9.0
exact = []
for e in E_VALUES:
    f, l = to_car_frame(LD, 0.0, 0.0, e, 0.0)
    exact = exact + [pursuit_wheel_angle(math.atan2(l, f), math.hypot(f, l))]
exact_deltas = np.array(exact)
linear_deltas = -equivalent_k_e(LD) * E_VALUES - equivalent_k_head(LD) * 0.0
relative_errors = np.abs(exact_deltas - linear_deltas) / np.abs(exact_deltas)
largest_linear_e = float(max(e for e, r in zip(E_VALUES, relative_errors) if r < 0.01))
gains_by_speed = {}
for speed in (10.0, 20.0, 30.0, 40.0):
    lookahead = lookahead_distance(speed)
    gains_by_speed[speed] = (equivalent_k_e(lookahead), equivalent_k_head(lookahead))
matching_speed = math.sqrt(2.0 * WHEELBASE / 0.2) / 0.3
detuning = (
    "Pure pursuit's lookahead grows with the speed, so its equivalent gains fall (K_E as 1/v², K_HEAD as 1/v): "
    "a faster car is steered more gently, and a gentle controller's late corrections do not overshoot "
    "and snowball the way a tight controller's do under a delay."
)

print("exact   ", exact_deltas)
print("linear  ", linear_deltas)
print("rel err ", relative_errors, "| largest e below 1%:", largest_linear_e)
for speed, (k_e, k_head) in gains_by_speed.items():
    print(f"v = {speed:4.0f} m/s: L_d = {lookahead_distance(speed):4.1f} m, K_E = {k_e:.4f} rad/m, K_HEAD = {k_head:.4f}")
print(f"K_E = 0.2 at v = {matching_speed:.4f} m/s")

# independent check: the closed form arctan(-2 L e / (L_d^2 + e^2))
closed_form = np.array([math.atan2(-2.0 * WHEELBASE * e, LD ** 2 + e ** 2) for e in E_VALUES])
assert np.max(np.abs(exact_deltas - closed_form)) < 1e-12
assert largest_linear_e == 0.5 and relative_errors[2] < 0.01 < relative_errors[3]
assert abs(gains_by_speed[30.0][0] - 0.0710) < 1e-4 and abs(gains_by_speed[30.0][1] - 0.639) < 1e-3  # the lesson's
assert abs(equivalent_k_e(lookahead_distance(matching_speed)) - 0.2) < 1e-12
assert abs(matching_speed - 17.8730) < 1e-4

## Exercise 5 - Stanley at the front axle

**Prediction.** Waypoint 255 of the `oval` is in the middle of a left bend of R = 120 m with waypoints about 2 m apart, so each segment turns about 2 / 120 ≈ 0.0167 rad further left than the one before.
The front axle is L = 2.875 m ahead along the car's heading, so it lands on the **next** segment, 256, whose heading is one step (+0.0167 rad) further left than the car's: ψ_e ≈ +0.0167.
A straight line along the tangent leaves a left bend to the outside, so the front axle on the line is a little to the **right** of the path (by a centimetre or so, about L² / (2R) less the chords' sag): e_f is slightly negative and both terms ask for a small left steer, about 0.02 rad, close to the bend's own wheel angle atan(L / R) ≈ 0.024 rad.

Shifted 0.5 m left (to the inside of the bend), the front axle's offset is a little under 0.5 m (the path bends toward it), the heading term is still +0.0167, and δ ≈ 0.0167 − arctan(1.5 / v), shrinking in size with the speed.
The **rear** axle reads the path where the car is, not ahead of it: the shifted point projects onto segment 254, the one **behind** (its clamped foot is a hair nearer than segment 255's endpoint), whose heading is one step to the right of the car's, so ψ_e ≈ −0.0167, and its offset is almost exactly 0.5 m.
So the rear-axle law turns about 2 × 0.0167 ≈ 0.033 rad **harder right** than the front-axle law, plus a little more for the larger offset: the front axle sees the bend coming and asks for less correction against it.
For small errors atan2(−k e, v) ≈ −(k / v)·e, so the gain at 30 m/s is 3 / 30 = 0.1 rad/m.

In [ ]:
def wrap_angle(angle):
    return (angle + math.pi) % (2.0 * math.pi) - math.pi


def project_onto_segment(px, py, ax, ay, bx, by):
    # Unit 2: (clamped foot t, squared distance to the clamped foot, signed distance to the line, left +)
    dx, dy = bx - ax, by - ay
    ex, ey = px - ax, py - ay
    seg2 = dx * dx + dy * dy
    t = min(max((ex * dx + ey * dy) / seg2, 0.0), 1.0)
    fx, fy = px - (ax + t * dx), py - (ay + t * dy)
    return t, fx * fx + fy * fy, (dx * ey - dy * ex) / math.sqrt(seg2)


def stanley_at(qx, qy, yaw, speed, xs, ys, segment, k):
    # Stanley's law with the cross-track error measured at the point (qx, qy).
    n = len(xs)
    best_d2, best_offset, best_segment = math.inf, 0.0, segment % n
    for index in range(segment - 4, segment + 12):
        a, b = index % n, (index + 1) % n
        _, d2, offset = project_onto_segment(qx, qy, xs[a], ys[a], xs[b], ys[b])
        if d2 < best_d2:
            best_d2, best_offset, best_segment = d2, offset, a
    a, b = best_segment, (best_segment + 1) % n
    heading_error = wrap_angle(math.atan2(ys[b] - ys[a], xs[b] - xs[a]) - yaw)
    return heading_error + math.atan2(-k * best_offset, max(speed, 1.0))


def stanley_wheel_angle(px, py, yaw, speed, xs, ys, segment, k=3.0):
    front_x, front_y = px + WHEELBASE * math.cos(yaw), py + WHEELBASE * math.sin(yaw)
    return stanley_at(front_x, front_y, yaw, speed, xs, ys, segment, k)


oval = load_track("oval")
oxs, oys = oval.x, oval.y
heading = math.atan2(oys[256] - oys[255], oxs[256] - oxs[255])
car_x, car_y = oxs[255], oys[255]
shifted_x, shifted_y = car_x - 0.5 * math.sin(heading), car_y + 0.5 * math.cos(heading)

delta_on_line = stanley_wheel_angle(car_x, car_y, heading, 20.0, oxs, oys, 255)
delta_left_by_speed = {speed: stanley_wheel_angle(shifted_x, shifted_y, heading, speed, oxs, oys, 255)
                       for speed in (5.0, 10.0, 20.0, 30.0)}
rear_axle_delta = stanley_at(shifted_x, shifted_y, heading, 20.0, oxs, oys, 255, 3.0)
gain_at_30 = 3.0 / 30.0
why_softer = (
    "The cross-track term is atan2(-k e, v), whose slope k / v falls as the car speeds up, so the same "
    "error asks for a smaller wheel angle at speed: like pure pursuit's speed-scaled lookahead in "
    "Exercise 4, Stanley's gain falls with v (as 1/v, while pure pursuit's K_E falls as 1/v²)."
)

print(f"on the line, 20 m/s: delta = {delta_on_line:+.6f} rad (the bend's own wheel angle atan(L/R) = "
      f"{math.atan2(WHEELBASE, 120.0):.6f})")
for speed, value in delta_left_by_speed.items():
    print(f"0.5 m left, v = {speed:4.0f} m/s: delta = {value:+.6f} rad")
print(f"rear axle, 0.5 m left, 20 m/s: delta = {rear_axle_delta:+.6f} rad "
      f"({rear_axle_delta - delta_left_by_speed[20.0]:+.4f} rad from the front-axle law); gain at 30 m/s {gain_at_30} rad/m")


# independent check: project on EVERY segment of the oval at once (NumPy), not on the window
def nearest_segment(qx, qy):
    ax, ay = oxs, oys
    bx, by = np.r_[oxs[1:], oxs[:1]], np.r_[oys[1:], oys[:1]]
    dx, dy = bx - ax, by - ay
    t = np.clip(((qx - ax) * dx + (qy - ay) * dy) / (dx * dx + dy * dy), 0.0, 1.0)
    d2 = (qx - ax - t * dx) ** 2 + (qy - ay - t * dy) ** 2
    i = int(np.argmin(d2))
    offset = (dx[i] * (qy - ay[i]) - dy[i] * (qx - ax[i])) / math.hypot(dx[i], dy[i])
    return i, wrap_angle(math.atan2(dy[i], dx[i]) - heading), offset


turn = wrap_angle(math.atan2(oys[257] - oys[256], oxs[257] - oxs[256]) - heading)
front = nearest_segment(shifted_x + WHEELBASE * math.cos(heading), shifted_y + WHEELBASE * math.sin(heading))
rear = nearest_segment(shifted_x, shifted_y)
on_line = nearest_segment(car_x + WHEELBASE * math.cos(heading), car_y + WHEELBASE * math.sin(heading))
assert front[0] == 256 and on_line[0] == 256 and rear[0] == 254
assert abs(front[1] - turn) < 1e-9 and abs(rear[1] + turn) < 1e-6 and abs(turn - 2.0 / 120.0) < 2e-4
assert abs(delta_on_line - (on_line[1] + math.atan2(-3.0 * on_line[2], 20.0))) < 1e-12
for speed, value in delta_left_by_speed.items():
    assert abs(value - (front[1] + math.atan2(-3.0 * front[2], speed))) < 1e-12
assert abs(rear_axle_delta - (rear[1] + math.atan2(-3.0 * rear[2], 20.0))) < 1e-12
assert -0.03 < on_line[2] < 0.0 and 0.0 < delta_on_line < math.atan2(WHEELBASE, 120.0)
assert 0.45 < front[2] < 0.5 and abs(rear[2] - 0.5) < 1e-3
assert rear_axle_delta < delta_left_by_speed[20.0] - 2.0 * turn  # the rear law steers harder right
assert abs(gain_at_30 * 0.5 - math.atan2(1.5, 30.0)) < 1e-3  # the small-error gain

## Exercise 6 - How much delay can pure pursuit take?

**Prediction**, from the lesson's study (this is the same setting): pure pursuit with k = 0.3 s is clean at every lag from 0 to 3; its RMS grows slowly to lag 2 (about 0.31 → 0.35 → 0.46 m) and then jumps at lag 3, where it weaves (about 1.4 m, about 13 s slower).
So the RMS should grow about 4.6 times, and the first lag more than 1 s slower is 3.

In [ ]:
pursuit_30 = write_variant(UNIT_05, workdir / "pursuit-30.py", {"TARGET_SPEED": "30.0"})
pursuit_by_lag = {}
for lag in range(4):
    result = run_race(pursuit_30, "test-circuit", car=KinematicCar(jitter=0.05, lag_ticks=lag),
                      seed=0, max_seconds=400.0)
    pursuit_by_lag[lag] = (clean(result), result.total_time, rms_offset(result))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(list(range(4)), [pursuit_by_lag[lag][2] for lag in range(4)], "o-")
ax.set_xlabel("actuator delay N (control ticks of 0.05 s)")
ax.set_ylabel("RMS cross-track error (m)")
ax.set_title("pure pursuit (PP_K = 0.3 s), test-circuit at 30 m/s, seed 0")
plt.show()

rms_growth = pursuit_by_lag[3][2] / pursuit_by_lag[0][2]
slow = [lag for lag in range(4) if pursuit_by_lag[lag][1] > pursuit_by_lag[0][1] + 1.0]
weaving_lag = slow[0] if slow else None
clean_but = (
    "Every race is clean, but at lag 3 the car is 13 s slower and four to five times further off the line: "
    "it weaves its way round, so 'clean' only says it stayed on the track, not that it was driven well."
)

for lag, (is_clean, time, rms) in pursuit_by_lag.items():
    print(f"lag {lag}: clean {is_clean}, {time:.2f} s, RMS {rms:.4f} m")
print(f"RMS growth lag 3 / lag 0: {rms_growth:.3f}; weaving from lag {weaving_lag}")

# the lesson's probe record for the same races
PINS = {0: (268.75, 0.3097), 1: (269.05, 0.3498), 2: (269.70, 0.4559), 3: (282.00, 1.4305)}
for lag, (time, rms) in PINS.items():
    assert pursuit_by_lag[lag][0] and abs(pursuit_by_lag[lag][1] - time) <= TICK
    assert abs(pursuit_by_lag[lag][2] - rms) <= RMS_TOL
assert weaving_lag == 3 and abs(rms_growth - 1.4305 / 0.3097) < 0.01

## Exercise 7 - Is Stanley better than Unit 4?

**Prediction.** The lesson's oval row is Unit 4 0.1041 m (range 0.0062) against Stanley 0.0987 m: a difference of about 0.005 m, **smaller** than Unit 4's range alone.
By Unit 1's rule that is not a difference beyond the noise.
Both controllers hold 20 m/s round the same line, so their race times should agree to a tick or so, again inside the noise.

In [ ]:
stanley = write_variant(UNIT_05, workdir / "stanley.py", {"STEERING": '"stanley"'})
unit_04_runs = repeat_races(UNIT_04, "oval", n=5, seeds=range(5), car_factory=lambda: KinematicCar(jitter=0.05))
stanley_runs = repeat_races(stanley, "oval", n=5, seeds=range(5), car_factory=lambda: KinematicCar(jitter=0.05))

unit_04_rms = [rms_offset(result) for result in unit_04_runs.results]
stanley_rms = [rms_offset(result) for result in stanley_runs.results]
rms_medians = {"unit-04": float(np.median(unit_04_rms)), "stanley": float(np.median(stanley_rms))}
rms_ranges = {"unit-04": max(unit_04_rms) - min(unit_04_rms), "stanley": max(stanley_rms) - min(stanley_rms)}
stanley_better_beyond_noise = rms_medians["unit-04"] - rms_medians["stanley"] > max(rms_ranges["unit-04"], rms_ranges["stanley"])
times_noise_aware = noise_aware_better(stanley_runs.times, unit_04_runs.times)
verdict = (
    "On the oval Stanley's median RMS is about 5 mm lower than Unit 4's, but that is less than either "
    "controller's seed-to-seed range, so we cannot claim it follows the line better; on the clock the two "
    "are within a tick of each other, so neither is faster beyond the noise."
)

print("unit-04 RMS:", np.round(unit_04_rms, 4), "| times", unit_04_runs.times)
print("stanley RMS:", np.round(stanley_rms, 4), "| times", stanley_runs.times)
print("medians", rms_medians, "ranges", rms_ranges)
print("Stanley's RMS better beyond the noise:", stanley_better_beyond_noise, "| faster beyond the noise:", times_noise_aware)

assert unit_04_runs.collisions == 0 and stanley_runs.collisions == 0
assert unit_04_runs.all_finished and stanley_runs.all_finished
assert abs(rms_medians["unit-04"] - 0.1041) <= RMS_TOL and abs(rms_ranges["unit-04"] - 0.0062) <= RMS_TOL  # the lesson's
assert abs(rms_medians["stanley"] - 0.0987) <= RMS_TOL
assert rms_medians["stanley"] < rms_medians["unit-04"] and not stanley_better_beyond_noise
assert not times_noise_aware

## Exercise 8 - Challenge: the best lookahead at lag 1

**Prediction.** One tick of delay is milder than two: the lesson's lag-1 row is clean for every controller and only about an eighth worse for pure pursuit.
So expect every race clean and the RMS to rise with k as at lag 0 (corner-cutting), with an optimum near the short end of the grid, perhaps 0.2 s, between the lag-0 and lag-2 optima.
That guess is the one to test: the grid here starts at 0.15 s, below anything the lesson's lag-0 sweep tried, and if the lowest RMS lands on an end of the grid, the sweep has not bracketed a minimum.
The k = 0.3 s race is Exercise 6's lag-1 race, so it is reused.

In [ ]:
K_GRID = (0.15, 0.2, 0.25, 0.3, 0.35, 0.4)
rms_by_k = {}
clean_by_k = {}
for k in K_GRID:
    if k == 0.3:
        is_clean, _, rms = pursuit_by_lag[1]
    else:
        source = write_variant(UNIT_05, workdir / f"pursuit-30-k{k}.py", {"TARGET_SPEED": "30.0", "PP_K": repr(k)})
        result = run_race(source, "test-circuit", car=KinematicCar(jitter=0.05, lag_ticks=1), seed=0, max_seconds=400.0)
        is_clean, rms = clean(result), rms_offset(result)
    rms_by_k[k] = rms
    clean_by_k[k] = is_clean
best_k_lag1 = min([k for k in K_GRID if clean_by_k[k]], key=lambda k: rms_by_k[k])
at_grid_edge = best_k_lag1 in (min(K_GRID), max(K_GRID))
extend_sweep = (
    "The best PP_K is the smallest one raced and the RMS still falls as PP_K shrinks, so the sweep has not found "
    "a minimum: extend it downwards (0.1, 0.05 s, ...) until the RMS turns back up (or a race stops being clean)."
)
one_seed_caveat = (
    "This is one seed of the jitter on one track; by Unit 1's rule I would race every PP_K on seeds 0-4 and "
    "call one better only if its median RMS beats the others by more than both ranges (after extending "
    "the grid below 0.15 s, since the best k sits at its edge)."
)

for k in K_GRID:
    print(f"PP_K = {k:.2f} s: clean {clean_by_k[k]}, RMS {rms_by_k[k]:.4f} m")
print("best at lag 1:", best_k_lag1, "| at the edge of the grid:", at_grid_edge)
print(extend_sweep)

assert all(clean_by_k[k] for k in K_GRID)
assert all(rms_by_k[a] < rms_by_k[b] for a, b in zip(K_GRID, K_GRID[1:]))  # monotone: corner-cutting dominates
assert best_k_lag1 == 0.15 and at_grid_edge and best_k_lag1 == min(K_GRID)

## Exercise 9 - Challenge: a target exactly L_d along the track

**Prediction.** On a straight the car sits 0.3 m left of waypoint i, the first waypoint at least 9 m away is i + 5 (10 m along, 2 m spacing), and the interpolated target is 9 m along.
The linearized law gives δ ≈ −2L·0.3 / d², so the two differ by about 2·2.875·0.3·(1/81 − 1/100) ≈ 0.0040 rad, and the straight-line distance overshoots L_d by about 1 m.
In a bend the chord is shorter than the arc, so the first-waypoint target can be further round the bend; the gap should be largest where the curvature changes, at the entry to the tightest corner, but still only a few thousandths of a radian.

In [ ]:
def interpolated_target(xs, ys, segment, px, py, lookahead):
    n = len(xs)
    a, b = segment % n, (segment + 1) % n
    dx, dy = xs[b] - xs[a], ys[b] - ys[a]
    t = min(max(((px - xs[a]) * dx + (py - ys[a]) * dy) / (dx * dx + dy * dy), 0.0), 1.0)
    fx, fy = xs[a] + t * dx, ys[a] + t * dy  # the clamped foot
    remaining = lookahead
    for _ in range(n):
        piece = math.hypot(xs[b] - fx, ys[b] - fy)
        if piece >= remaining:
            u = remaining / piece
            return fx + u * (xs[b] - fx), fy + u * (ys[b] - fy)
        remaining = remaining - piece
        fx, fy = xs[b], ys[b]
        b = (b + 1) % n
    return fx, fy


def wheel_angle_to(tx, ty, cx, cy, yaw):
    f, l = to_car_frame(tx, ty, cx, cy, yaw)
    return pursuit_wheel_angle(math.atan2(l, f), math.hypot(f, l))


n_waypoints = len(xs)
POSES = list(range(0, n_waypoints, 10))
first_angles, interpolated_angles, extra = [], [], []
for i in POSES:
    j = (i + 1) % n_waypoints
    pose_yaw = math.atan2(ys[j] - ys[i], xs[j] - xs[i])
    cx, cy = xs[i] - 0.3 * math.sin(pose_yaw), ys[i] + 0.3 * math.cos(pose_yaw)
    k = first_target(xs, ys, i, cx, cy, 9.0)
    tx, ty = interpolated_target(xs, ys, i, cx, cy, 9.0)
    first_angles = first_angles + [wheel_angle_to(xs[k], ys[k], cx, cy, pose_yaw)]
    interpolated_angles = interpolated_angles + [wheel_angle_to(tx, ty, cx, cy, pose_yaw)]
    extra = extra + [math.hypot(xs[k] - cx, ys[k] - cy) - 9.0]
delta_first = np.array(first_angles)
delta_interpolated = np.array(interpolated_angles)
gaps_9 = np.abs(delta_first - delta_interpolated)
max_delta_gap = float(np.max(gaps_9))
mean_extra_distance = float(np.mean(extra))
worst_pose = POSES[int(np.argmax(gaps_9))]
worth_it = (
    f"No: the largest gap is {max_delta_gap:.4f} rad (about {math.degrees(max_delta_gap):.2f} degrees), at "
    f"waypoint {worst_pose}, the entry to the tightest corner, and almost everywhere else it is the "
    "same 0.004 rad that the extra 1 m of lookahead costs on a straight; the effect is the same as a "
    "slightly longer lookahead (about 10 m instead of 9), which PP_K can absorb."
)

print(f"{len(POSES)} poses; max gap {max_delta_gap:.6f} rad at waypoint {worst_pose}; "
      f"mean extra distance {mean_extra_distance:.4f} m")
print(worth_it)

# independent checks: the interpolated target is exactly L_d along a straight from the start pose
tx, ty = interpolated_target(xs, ys, 0, xs[0] - 0.3 * math.sin(yaw), ys[0] + 0.3 * math.cos(yaw), 9.0)
assert abs(math.hypot(tx - xs[0], ty - ys[0]) - 9.0) < 1e-6
straight_gap = 2.0 * WHEELBASE * 0.3 * (1.0 / 81.0 - 1.0 / 100.0)
assert abs(np.median(gaps_9) - straight_gap) < 2e-4
assert len(POSES) == 134 and max_delta_gap < 0.01
assert abs(mean_extra_distance - 1.0) < 0.05